# Transformer All Tasks —  Baseline

This notebook trains Standard Transformer and Set Transformer task models using the final hyperparameters copied from the Transformer fine-tuning notebook.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing 'src'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [2]:
import pandas as pd
import torch

from src.data_prep import prepare_uji_data
from src.training import TrainConfig, train_from_tensors
from src.models.set_transformer import (
    CoordinateSetTransformerModel,
    JointSetTransformerModel,
    MultiTaskSetTransformerModel,
    SetTransformerConfig,
)
from src.models.standard_transformer import (
    CoordinateTransformerModel,
    JointTransformerModel,
    MultiTaskTransformerModel,
    TransformerConfig,
)


In [3]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = bundle.X_train.shape[1]

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)
print("coordinate_std:", bundle.coordinate_std)


device: cuda
X train/val: (19937, 1040) (1111, 1040)
coordinate_std: [123.39891  66.94215]


## Paste final fine-tuned settings

Copy the `FINAL_TUNED_CFGS` dictionary from the matching fine-tuning notebook and paste it into the cell below.

This is intentionally manual. It makes the baseline notebook self-contained and avoids silently depending on CSV files that may be missing, stale, or outside the submitted repo.


In [4]:
# Paste the FINAL_TUNED_CFGS dictionary printed by fair_transformer_hyperparameter_tuning_v3_original_names.ipynb here.
#
# Expected shape from the fine-tuning notebook:
# FINAL_TUNED_CFGS = {
#     "standard": {
#         "joint": {
#             "lr": ..., "weight_decay": ..., "dropout": ...,
#             "grad_clip_norm": ..., "max_epochs": ..., "patience": ...,
#             "print_every": 5, "batch_size": 256, "val_batch_size": 512,
#             "architecture": {"d_model": 128, "nhead": 4, "num_layers": 2, "dim_feedforward": 256},
#         },
#         "multitask": {...},
#         "coordinate": {...},
#     },
#     "set": {
#         "joint": {"architecture": {"d_model": 128, "num_heads": 4, "num_sab_layers": 2, "dim_feedforward": 256, "num_seed_vectors": 1}, ...},
#         ...
#     }
# }

FINAL_TUNED_CFGS = {'set': {'coordinate': {'lr': 0.001,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'num_heads': 4,
    'num_sab_layers': 2,
    'dim_feedforward': 256,
    'num_seed_vectors': 1}},
  'joint': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'num_heads': 4,
    'num_sab_layers': 2,
    'dim_feedforward': 256,
    'num_seed_vectors': 1}},
  'multitask': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'num_heads': 4,
    'num_sab_layers': 2,
    'dim_feedforward': 256,
    'num_seed_vectors': 1}}},
 'standard': {'coordinate': {'lr': 0.001,
   'weight_decay': 0.0001,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'nhead': 4,
    'num_layers': 2,
    'dim_feedforward': 256}},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'nhead': 4,
    'num_layers': 2,
    'dim_feedforward': 256}},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0001,
   'dropout': 0.1,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512,
   'architecture': {'d_model': 128,
    'nhead': 4,
    'num_layers': 2,
    'dim_feedforward': 256}}}}

TASKS = ("joint", "multitask", "coordinate")
TRANSFORMER_FAMILIES = ("standard", "set")
TRAIN_KEYS = ("lr", "weight_decay", "max_epochs", "patience", "print_every", "batch_size", "val_batch_size", "grad_clip_norm")

DEFAULT_STANDARD_ARCH = dict(d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.1)
DEFAULT_SET_ARCH = dict(d_model=128, num_heads=4, num_sab_layers=2, dim_feedforward=256, num_seed_vectors=1, dropout=0.1)

def _entry_for(family: str, task: str) -> dict:
    return dict(FINAL_TUNED_CFGS[family][task])

def _train_spec(entry: dict) -> dict:
    # Supports both shapes:
    # 1) flat tuning output: {"lr": ..., "architecture": {...}}
    # 2) nested baseline output: {"train": {...}, "architecture": {...}}
    raw = dict(entry.get("train", entry))
    raw.pop("architecture", None)
    raw.pop("dropout", None)
    return {k: raw.get(k) for k in TRAIN_KEYS if k in raw}

def _architecture_spec(family: str, entry: dict) -> dict:
    if family == "standard":
        arch = dict(DEFAULT_STANDARD_ARCH)
    elif family == "set":
        arch = dict(DEFAULT_SET_ARCH)
    else:
        raise ValueError(f"Unknown transformer family: {family}")

    arch.update(dict(entry.get("architecture", {})))
    if "dropout" in entry:
        arch["dropout"] = entry["dropout"]
    if "train" in entry and "dropout" in entry["train"]:
        arch["dropout"] = entry["train"]["dropout"]
    return arch

def validate_manual_transformer_configs(final_cfgs: dict) -> pd.DataFrame:
    if not final_cfgs:
        raise RuntimeError(
            "FINAL_TUNED_CFGS is empty. Copy the final dictionary from fair_transformer_hyperparameter_tuning_v3_original_names.ipynb first."
        )

    missing = []
    rows = []
    for family in TRANSFORMER_FAMILIES:
        if family not in final_cfgs:
            missing.append((family, "<family missing>"))
            continue
        for task in TASKS:
            if task not in final_cfgs[family]:
                missing.append((family, task))
                continue
            entry = _entry_for(family, task)
            train = _train_spec(entry)
            arch = _architecture_spec(family, entry)
            missing_train = [k for k in ("lr", "weight_decay", "max_epochs", "patience") if k not in train or train[k] is None]
            if missing_train:
                raise RuntimeError(f"Train config for {(family, task)} is missing keys: {missing_train}")
            rows.append({"family": family, "task": task, **train, **arch})

    if missing:
        raise RuntimeError(f"Missing required tuned configs: {missing}")

    return pd.DataFrame(rows).sort_values(["family", "task"]).reset_index(drop=True)

selected_cfg_df = validate_manual_transformer_configs(FINAL_TUNED_CFGS)
selected_cfg_df


,family,task,lr,weight_decay,max_epochs,patience,print_every,batch_size,val_batch_size,grad_clip_norm,d_model,nhead,num_layers,dim_feedforward,dropout,num_heads,num_sab_layers,num_seed_vectors
0,set,coordinate,0.0010,0.0005,50,10,5,256,512,1.0,128,NaN,NaN,256,0.1,4.0,2.0,1.0
1,set,joint,0.0005,0.0005,80,15,5,256,512,1.0,128,NaN,NaN,256,0.1,4.0,2.0,1.0
2,set,multitask,0.0005,0.0005,80,15,5,256,512,1.0,128,NaN,NaN,256,0.1,4.0,2.0,1.0
3,standard,coordinate,0.0010,0.0001,50,10,5,256,512,1.0,128,4.0,2.0,256,0.1,NaN,NaN,NaN
4,standard,joint,0.0010,0.0005,50,10,5,256,512,1.0,128,4.0,2.0,256,0.1,NaN,NaN,NaN
5,standard,multitask,0.0010,0.0001,50,10,5,256,512,1.0,128,4.0,2.0,256,0.1,NaN,NaN,NaN


In [5]:
def make_train_cfg(family: str, task: str, run_name: str) -> TrainConfig:
    entry = _entry_for(family, task)
    spec = _train_spec(entry)
    spec["run_name"] = run_name
    return TrainConfig(**spec)

def make_standard_cfg(task: str) -> TransformerConfig:
    return TransformerConfig(**_architecture_spec("standard", _entry_for("standard", task)))

def make_set_cfg(task: str) -> SetTransformerConfig:
    return SetTransformerConfig(**_architecture_spec("set", _entry_for("set", task)))

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def result_row(name, model, result):
    row = {
        "model": name,
        "params": count_trainable_params(model),
        "best_epoch": result.best_epoch,
    }
    row.update(result.best_metrics)
    return row


## Train joint-classification models with imported tuned settings

In [6]:
joint_transformer = JointTransformerModel(
    in_dim=in_dim,
    backbone_cfg=make_standard_cfg("joint"),
)
joint_transformer_result = train_from_tensors(
    model=joint_transformer,
    X_train=bundle.X_train,
    y_train=joint_y_train,
    X_val=bundle.X_val,
    y_val=joint_y_val,
    device=device,
    cfg=make_train_cfg("standard", "joint", "transformer_joint_baseline"),
)

joint_set_transformer = JointSetTransformerModel(
    in_dim=in_dim,
    backbone_cfg=make_set_cfg("joint"),
)
joint_set_transformer_result = train_from_tensors(
    model=joint_set_transformer,
    X_train=bundle.X_train,
    y_train=joint_y_train,
    X_val=bundle.X_val,
    y_val=joint_y_val,
    device=device,
    cfg=make_train_cfg("set", "joint", "set_transformer_joint_baseline"),
)

pd.DataFrame([
    result_row("transformer_joint", joint_transformer, joint_transformer_result),
    result_row("set_transformer_joint", joint_set_transformer, joint_set_transformer_result),
])


epoch=001 train_loss=2.4952 val_loss=2.1858 score=0.2124
epoch=005 train_loss=0.5349 val_loss=0.4314 score=0.8668
epoch=010 train_loss=0.1285 val_loss=0.2365 score=0.9469
epoch=015 train_loss=0.0612 val_loss=0.2621 score=0.9343
epoch=020 train_loss=0.0278 val_loss=0.2366 score=0.9478
epoch=001 train_loss=2.4949 val_loss=2.4253 score=0.0783
epoch=005 train_loss=1.1773 val_loss=0.9462 score=0.7075
epoch=010 train_loss=0.5301 val_loss=0.5272 score=0.8317
epoch=015 train_loss=0.2712 val_loss=0.3156 score=0.9163
epoch=020 train_loss=0.2425 val_loss=0.2902 score=0.9307
epoch=025 train_loss=0.1272 val_loss=0.2993 score=0.9271
epoch=030 train_loss=0.1033 val_loss=0.2238 score=0.9523
epoch=035 train_loss=0.0874 val_loss=0.2283 score=0.9577
epoch=040 train_loss=0.0518 val_loss=0.3708 score=0.9127
epoch=045 train_loss=0.0295 val_loss=0.3070 score=0.9379
epoch=050 train_loss=0.0227 val_loss=0.2829 score=0.9559


,model,params,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy
0,transformer_joint,350861,12,12.0,0.089042,0.216645,0.953195,0.953195,0.9982,0.954095
1,set_transformer_joint,483213,35,35.0,0.087419,0.228287,0.957696,0.957696,1.0000,0.957696


## Train multi-task models with imported tuned settings

In [7]:
multitask_transformer = MultiTaskTransformerModel(
    in_dim=in_dim,
    backbone_cfg=make_standard_cfg("multitask"),
)
multitask_transformer_result = train_from_tensors(
    model=multitask_transformer,
    X_train=bundle.X_train,
    y_train=mt_y_train,
    X_val=bundle.X_val,
    y_val=mt_y_val,
    device=device,
    cfg=make_train_cfg("standard", "multitask", "transformer_multitask_baseline"),
)

multitask_set_transformer = MultiTaskSetTransformerModel(
    in_dim=in_dim,
    backbone_cfg=make_set_cfg("multitask"),
)
multitask_set_transformer_result = train_from_tensors(
    model=multitask_set_transformer,
    X_train=bundle.X_train,
    y_train=mt_y_train,
    X_val=bundle.X_val,
    y_val=mt_y_val,
    device=device,
    cfg=make_train_cfg("set", "multitask", "set_transformer_multitask_baseline"),
)

pd.DataFrame([
    result_row("transformer_multitask", multitask_transformer, multitask_transformer_result),
    result_row("set_transformer_multitask", multitask_set_transformer, multitask_set_transformer_result),
])


epoch=001 train_loss=2.4392 val_loss=2.1076 score=0.0981
epoch=005 train_loss=0.8339 val_loss=0.6582 score=0.7003
epoch=010 train_loss=0.5093 val_loss=0.4466 score=0.8488
epoch=015 train_loss=0.3681 val_loss=0.4210 score=0.8929
epoch=020 train_loss=0.2006 val_loss=0.3677 score=0.9217
epoch=025 train_loss=0.1029 val_loss=0.4054 score=0.9145
epoch=030 train_loss=0.0584 val_loss=0.2781 score=0.9415
epoch=035 train_loss=0.0496 val_loss=0.2585 score=0.9487
epoch=040 train_loss=0.0350 val_loss=0.2966 score=0.9460
epoch=045 train_loss=0.0214 val_loss=0.3249 score=0.9478
epoch=001 train_loss=2.5604 val_loss=2.3936 score=0.1098
epoch=005 train_loss=1.1765 val_loss=0.8469 score=0.6958
epoch=010 train_loss=0.5529 val_loss=0.5295 score=0.7840
epoch=015 train_loss=0.3010 val_loss=0.4935 score=0.8209
epoch=020 train_loss=0.1186 val_loss=0.3444 score=0.9181
epoch=025 train_loss=0.0899 val_loss=0.3417 score=0.9280
epoch=030 train_loss=0.0398 val_loss=0.3669 score=0.9316
epoch=035 train_loss=0.0278 val

,model,params,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy
0,transformer_multitask,366728,36,36.0,0.034458,0.237330,0.955896,0.955896,0.9991,0.955896
1,set_transformer_multitask,499080,22,22.0,0.135010,0.247083,0.945095,0.945095,0.9964,0.946895


## Train coordinate-regression models with imported tuned settings

In [8]:
coord_transformer = CoordinateTransformerModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
    backbone_cfg=make_standard_cfg("coordinate"),
)
coord_transformer_result = train_from_tensors(
    model=coord_transformer,
    X_train=bundle.X_train,
    y_train=coord_y_train,
    X_val=bundle.X_val,
    y_val=coord_y_val,
    device=device,
    cfg=make_train_cfg("standard", "coordinate", "transformer_coordinate_baseline"),
)

coord_set_transformer = CoordinateSetTransformerModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
    backbone_cfg=make_set_cfg("coordinate"),
)
coord_set_transformer_result = train_from_tensors(
    model=coord_set_transformer,
    X_train=bundle.X_train,
    y_train=coord_y_train,
    X_val=bundle.X_val,
    y_val=coord_y_val,
    device=device,
    cfg=make_train_cfg("set", "coordinate", "set_transformer_coordinate_baseline"),
)

pd.DataFrame([
    result_row("transformer_coordinate", coord_transformer, coord_transformer_result),
    result_row("set_transformer_coordinate", coord_set_transformer, coord_set_transformer_result),
])


epoch=001 train_loss=0.7415 val_loss=0.2342 score=-55.2104
epoch=005 train_loss=0.0526 val_loss=0.0414 score=-23.4090
epoch=010 train_loss=0.0184 val_loss=0.0174 score=-12.6623
epoch=015 train_loss=0.0154 val_loss=0.0164 score=-13.3299
epoch=020 train_loss=0.0148 val_loss=0.0156 score=-11.3816
epoch=025 train_loss=0.0121 val_loss=0.0151 score=-11.4600
epoch=030 train_loss=0.0119 val_loss=0.0149 score=-11.2600
epoch=035 train_loss=0.0113 val_loss=0.0148 score=-10.7640
epoch=040 train_loss=0.0107 val_loss=0.0148 score=-10.6004
epoch=045 train_loss=0.0105 val_loss=0.0142 score=-10.4113
epoch=001 train_loss=0.6413 val_loss=0.1781 score=-47.5806
epoch=005 train_loss=0.0632 val_loss=0.0552 score=-26.5435
epoch=010 train_loss=0.0227 val_loss=0.0167 score=-13.6498
epoch=015 train_loss=0.0155 val_loss=0.0167 score=-13.8208
epoch=020 train_loss=0.0131 val_loss=0.0127 score=-10.9001
epoch=025 train_loss=0.0121 val_loss=0.0143 score=-11.9830
epoch=030 train_loss=0.0116 val_loss=0.0130 score=-11.81

,model,params,best_epoch,epoch,train_loss,val_loss,score,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,transformer_coordinate,349442,39,39.0,0.010772,0.014639,-10.340784,0.120322,0.165203,10.340784,13.908373
1,set_transformer_coordinate,481794,44,44.0,0.009577,0.011353,-9.764674,0.112004,0.149488,9.764674,12.678125


## Final comparison table

In [9]:
transformer_comparison_df = pd.DataFrame([
    result_row("transformer_joint", joint_transformer, joint_transformer_result),
    result_row("set_transformer_joint", joint_set_transformer, joint_set_transformer_result),
    result_row("transformer_multitask", multitask_transformer, multitask_transformer_result),
    result_row("set_transformer_multitask", multitask_set_transformer, multitask_set_transformer_result),
    result_row("transformer_coordinate", coord_transformer, coord_transformer_result),
    result_row("set_transformer_coordinate", coord_set_transformer, coord_set_transformer_result),
])
transformer_comparison_df


,model,params,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,transformer_joint,350861,12,12.0,0.089042,0.216645,0.953195,0.953195,0.9982,0.954095,NaN,NaN,NaN,NaN
1,set_transformer_joint,483213,35,35.0,0.087419,0.228287,0.957696,0.957696,1.0000,0.957696,NaN,NaN,NaN,NaN
2,transformer_multitask,366728,36,36.0,0.034458,0.237330,0.955896,0.955896,0.9991,0.955896,NaN,NaN,NaN,NaN
3,set_transformer_multitask,499080,22,22.0,0.135010,0.247083,0.945095,0.945095,0.9964,0.946895,NaN,NaN,NaN,NaN
4,transformer_coordinate,349442,39,39.0,0.010772,0.014639,-10.340784,NaN,NaN,NaN,0.120322,0.165203,10.340784,13.908373
5,set_transformer_coordinate,481794,44,44.0,0.009577,0.011353,-9.764674,NaN,NaN,NaN,0.112004,0.149488,9.764674,12.678125


## Parameter-count report

In [10]:
transformer_param_df = pd.DataFrame([
    {"model": "transformer_joint", "params": count_trainable_params(joint_transformer)},
    {"model": "transformer_multitask", "params": count_trainable_params(multitask_transformer)},
    {"model": "transformer_coordinate", "params": count_trainable_params(coord_transformer)},
    {"model": "set_transformer_joint", "params": count_trainable_params(joint_set_transformer)},
    {"model": "set_transformer_multitask", "params": count_trainable_params(multitask_set_transformer)},
    {"model": "set_transformer_coordinate", "params": count_trainable_params(coord_set_transformer)},
])
transformer_param_df["params_millions"] = transformer_param_df["params"] / 1_000_000
transformer_param_df


,model,params,params_millions
0,transformer_joint,350861,0.350861
1,transformer_multitask,366728,0.366728
2,transformer_coordinate,349442,0.349442
3,set_transformer_joint,483213,0.483213
4,set_transformer_multitask,499080,0.499080
5,set_transformer_coordinate,481794,0.481794


## Latency benchmark

In [11]:
import time
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class LatencyConfig:
    device: str = "cpu"
    batch_size: int = 1
    n_samples: int = 512
    n_warmup: int = 50
    n_repeats: int = 3
    seed: int = 42
    percentile: float = 95.0

def count_trainable_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def _synchronize_if_needed(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def _select_shared_subset(X, n_samples: int, seed: int) -> np.ndarray:
    n_total = int(len(X))
    if n_total == 0:
        raise ValueError("X is empty.")
    n_use = min(int(n_samples), n_total)
    rng = np.random.default_rng(seed)
    return rng.choice(n_total, size=n_use, replace=False)

def benchmark_single_model_latency(model: torch.nn.Module, X, cfg: LatencyConfig) -> dict[str, float]:
    if cfg.batch_size != 1:
        raise ValueError("This helper is designed for batch_size=1 fair-comparison latency.")

    device = torch.device(cfg.device)
    model = model.to(device)
    model.eval()

    X_t = torch.as_tensor(X, dtype=torch.float32)
    subset_idx = _select_shared_subset(X_t, cfg.n_samples, cfg.seed)
    X_subset = X_t[subset_idx]

    with torch.inference_mode():
        for i in range(min(cfg.n_warmup, len(X_subset))):
            xb = X_subset[i:i+1].to(device, non_blocking=False)
            _ = model(xb)
        _synchronize_if_needed(device)

    per_sample_times_ms = []
    with torch.inference_mode():
        for _rep in range(cfg.n_repeats):
            for i in range(len(X_subset)):
                xb = X_subset[i:i+1].to(device, non_blocking=False)
                _synchronize_if_needed(device)
                t0 = time.perf_counter()
                _ = model(xb)
                _synchronize_if_needed(device)
                t1 = time.perf_counter()
                per_sample_times_ms.append((t1 - t0) * 1000.0)

    arr = np.asarray(per_sample_times_ms, dtype=np.float64)
    return {
        "param_count": int(count_trainable_params(model)),
        "n_samples": int(len(X_subset)),
        "n_runs": int(len(arr)),
        "mean_ms": float(arr.mean()),
        "median_ms": float(np.median(arr)),
        "p95_ms": float(np.percentile(arr, cfg.percentile)),
        "std_ms": float(arr.std(ddof=0)),
        "min_ms": float(arr.min()),
        "max_ms": float(arr.max()),
        **{f"latency_cfg_{k}": v for k, v in asdict(cfg).items()},
    }

def benchmark_model_dict(model_dict: dict[str, torch.nn.Module], X, cfg: LatencyConfig) -> pd.DataFrame:
    rows = []
    for name, model in model_dict.items():
        print(f"Benchmarking {name}...")
        row = {"model": name}
        row.update(benchmark_single_model_latency(model, X, cfg))
        rows.append(row)
    return pd.DataFrame(rows)


In [12]:
latency_cfg = LatencyConfig(
    device="cpu",
    batch_size=1,
    n_samples=512,
    n_warmup=50,
    n_repeats=3,
    seed=42,
    percentile=95.0,
)

transformer_latency_df = benchmark_model_dict(
    model_dict={
        "transformer_joint": joint_transformer,
        "transformer_multitask": multitask_transformer,
        "transformer_coordinate": coord_transformer,
        "set_transformer_joint": joint_set_transformer,
        "set_transformer_multitask": multitask_set_transformer,
        "set_transformer_coordinate": coord_set_transformer,
    },
    X=bundle.X_val,
    cfg=latency_cfg,
)

transformer_latency_df


Benchmarking transformer_joint...
Benchmarking transformer_multitask...
Benchmarking transformer_coordinate...
Benchmarking set_transformer_joint...
Benchmarking set_transformer_multitask...
Benchmarking set_transformer_coordinate...


,model,param_count,n_samples,n_runs,mean_ms,median_ms,p95_ms,std_ms,min_ms,max_ms,latency_cfg_device,latency_cfg_batch_size,latency_cfg_n_samples,latency_cfg_n_warmup,latency_cfg_n_repeats,latency_cfg_seed,latency_cfg_percentile
0,transformer_joint,350861,512,1536,38.275079,33.887397,85.636717,27.640028,2.984741,396.985833,cpu,1,512,50,3,42,95.0
1,transformer_multitask,366728,512,1536,68.935193,58.959631,143.373112,45.545513,3.671755,586.530894,cpu,1,512,50,3,42,95.0
2,transformer_coordinate,349442,512,1536,40.958579,33.144508,108.772175,36.849854,2.613948,398.939592,cpu,1,512,50,3,42,95.0
3,set_transformer_joint,483213,512,1536,97.941671,86.236373,182.389657,65.093159,4.704424,563.320785,cpu,1,512,50,3,42,95.0
4,set_transformer_multitask,499080,512,1536,165.066986,155.500977,285.100562,66.534833,34.295505,527.058588,cpu,1,512,50,3,42,95.0
5,set_transformer_coordinate,481794,512,1536,124.094846,117.076287,231.853709,108.091119,4.664899,3077.702305,cpu,1,512,50,3,42,95.0


## Save outputs

In [13]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "logs" / "fair_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

transformer_comparison_df.to_csv(OUTPUT_DIR / "transformer_baseline_results.csv", index=False)
transformer_param_df.to_csv(OUTPUT_DIR / "transformer_parameter_report.csv", index=False)
transformer_latency_df.to_csv(OUTPUT_DIR / "transformer_latency_results.csv", index=False)

print("Saved baseline outputs to:", OUTPUT_DIR)


Saved baseline outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/fair_baseline


## Save Checkpoint

In [14]:
from pathlib import Path
import torch

# Save under project-level models/ directory
MODEL_DIR = Path("../models") if Path.cwd().name == "notebooks" else Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving checkpoints to: {MODEL_DIR.resolve()}")

def tensor_to_list(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    return x

def maybe_to_dict(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if hasattr(x, "__dict__"):
        return dict(x.__dict__)
    return {"value": x}

def safe_best_metrics(result_obj):
    return maybe_to_dict(getattr(result_obj, "best_metrics", {}))

def safe_best_epoch(result_obj):
    return getattr(result_obj, "best_epoch", None)

def save_checkpoint(
    path,
    model,
    model_class_name,
    task,
    in_dim,
    model_kwargs=None,
    train_cfg=None,
    best_metrics=None,
    best_epoch=None,
):
    payload = {
        "model_class_name": model_class_name,
        "task": task,
        "in_dim": in_dim,
        "model_kwargs": model_kwargs or {},
        "train_cfg": train_cfg or {},
        "best_metrics": best_metrics or {},
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }
    torch.save(payload, path)
    print(f"Saved checkpoint: {path}")

Saving checkpoints to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/models


In [ ]:
# -----------------------------
# Standard Transformer checkpoints
# -----------------------------
save_checkpoint(
    MODEL_DIR / "transformer_joint.pt",
    joint_transformer,
    model_class_name=type(joint_transformer).__name__,
    task="joint",
    in_dim=in_dim,
    model_kwargs={
        "backbone_cfg": maybe_to_dict(make_standard_cfg("joint")),
    },
    best_metrics=safe_best_metrics(joint_transformer_result),
    best_epoch=safe_best_epoch(joint_transformer_result),
)

save_checkpoint(
    MODEL_DIR / "transformer_multitask.pt",
    multitask_transformer,
    model_class_name=type(multitask_transformer).__name__,
    task="multitask",
    in_dim=in_dim,
    model_kwargs={
        "backbone_cfg": maybe_to_dict(make_standard_cfg("multitask")),
    },
    best_metrics=safe_best_metrics(multitask_transformer_result),
    best_epoch=safe_best_epoch(multitask_transformer_result),
)

save_checkpoint(
    MODEL_DIR / "transformer_coordinate.pt",
    coord_transformer,
    model_class_name=type(coord_transformer).__name__,
    task="coordinate",
    in_dim=in_dim,
    model_kwargs={
        "coordinate_std": tensor_to_list(bundle.coordinate_std),
        "backbone_cfg": maybe_to_dict(make_standard_cfg("coordinate")),
    },
    best_metrics=safe_best_metrics(coord_transformer_result),
    best_epoch=safe_best_epoch(coord_transformer_result),
)

# -----------------------------
# Set Transformer checkpoints
# -----------------------------
save_checkpoint(
    MODEL_DIR / "set_transformer_joint.pt",
    joint_set_transformer,
    model_class_name=type(joint_set_transformer).__name__,
    task="joint",
    in_dim=in_dim,
    model_kwargs={
        "backbone_cfg": maybe_to_dict(make_set_cfg("joint")),
    },
    best_metrics=safe_best_metrics(joint_set_transformer_result),
    best_epoch=safe_best_epoch(joint_set_transformer_result),
)

save_checkpoint(
    MODEL_DIR / "set_transformer_multitask.pt",
    multitask_set_transformer,
    model_class_name=type(multitask_set_transformer).__name__,
    task="multitask",
    in_dim=in_dim,
    model_kwargs={
        "backbone_cfg": maybe_to_dict(make_set_cfg("multitask")),
    },
    best_metrics=safe_best_metrics(multitask_set_transformer_result),
    best_epoch=safe_best_epoch(multitask_set_transformer_result),
)

save_checkpoint(
    MODEL_DIR / "set_transformer_coordinate.pt",
    coord_set_transformer,
    model_class_name=type(coord_set_transformer).__name__,
    task="coordinate",
    in_dim=in_dim,
    model_kwargs={
        "coordinate_std": tensor_to_list(bundle.coordinate_std),
        "backbone_cfg": maybe_to_dict(make_set_cfg("coordinate")),
    },
    best_metrics=safe_best_metrics(coord_set_transformer_result),
    best_epoch=safe_best_epoch(coord_set_transformer_result),
)

Saved checkpoint: ../models/transformer_joint.pt
Saved checkpoint: ../models/transformer_multitask.pt
Saved checkpoint: ../models/transformer_coordinate.pt
Saved checkpoint: ../models/set_transformer_joint.pt
Saved checkpoint: ../models/set_transformer_multitask.pt
Saved checkpoint: ../models/set_transformer_coordinate.pt


: 